In [1]:
import pandas as pd
import numpy as np
import os
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import cohen_kappa_score



In [2]:
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

train_df1 = pd.read_csv(
    "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/train.csv"
)



/kaggle/input/description.md
/kaggle/input/sample_submission.csv
/kaggle/input/sample_submission.csv.zip
/kaggle/input/test.csv
/kaggle/input/train.csv.zip
/kaggle/input/train.csv
/kaggle/input/test.csv.zip
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/description.md
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/sample_submission.csv
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/sample_submission.csv.zip
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/train.csv.zip
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/train.csv
/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv.zip


In [3]:
train_df1["score"].dtypes



dtype('int64')

In [4]:
train_df1.head(10)



,essay_id,full_text,score
0,663d2cf,"Dear State Senator,\n\nI am arguing in favor o...",3
1,3a20bfb,"In "" The Challenge of Exploring Venus"" The aut...",2
2,6adae64,Teachers can have a hard time telling if their...,3
3,c81ccdf,Using dirverless cars can be very dangrous in ...,3
4,9549d7f,"In "" The Challenge of Exploring Venus"" the aut...",3
5,15992f4,What would the world be like if we could just ...,3
6,1b7e42c,Alien Landform?\n\nDo you think that the face ...,1
7,e06d5d9,"My argument is for ""Driverless cars"", the reas...",3
8,a779bc6,Have you ever seen those movies will the smart...,4
9,ca7b3c6,"Dont you get tired of driving sometimes, Get s...",2


In [5]:
train_df1["score"].value_counts()



score
3    5629
2    4249
4    3563
1    1124
5     876
6     135
Name: count, dtype: int64

In [6]:
submission = pd.read_csv(
    "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/sample_submission.csv"
)
submission



,essay_id,score
0,d550b2d,4
1,0c10954,5
2,ef04816,3
3,88ab8d9,5
4,cee8c0f,5
...,...,...
1726,d8c9e3d,3
1727,6c3e2b9,5
1728,743f56b,5
1729,03630c7,1


In [7]:
cList = {
    "ain't": "am not",
    "aren't": "are not",
    "can't": "cannot",
    "can't've": "cannot have",
    "'cause": "because",
    "could've": "could have",
    "couldn't": "could not",
    "couldn't've": "could not have",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hadn't've": "had not have",
    "hasn't": "has not",
    "haven't": "have not",
    "he'd": "he would",
    "he'd've": "he would have",
    "he'll": "he will",
    "he'll've": "he will have",
    "he's": "he is",
    "how'd": "how did",
    "how'd'y": "how do you",
    "how'll": "how will",
    "how's": "how is",
    "I'd": "I would",
    "I'd've": "I would have",
    "I'll": "I will",
    "I'll've": "I will have",
    "I'm": "I am",
    "I've": "I have",
    "isn't": "is not",
    "it'd": "it had",
    "it'd've": "it would have",
    "it'll": "it will",
    "it'll've": "it will have",
    "it's": "it is",
    "let's": "let us",
    "ma'am": "madam",
    "mayn't": "may not",
    "might've": "might have",
    "mightn't": "might not",
    "mightn't've": "might not have",
    "must've": "must have",
    "mustn't": "must not",
    "mustn't've": "must not have",
    "needn't": "need not",
    "needn't've": "need not have",
    "o'clock": "of the clock",
    "oughtn't": "ought not",
    "oughtn't've": "ought not have",
    "shan't": "shall not",
    "sha'n't": "shall not",
    "shan't've": "shall not have",
    "she'd": "she would",
    "she'd've": "she would have",
    "she'll": "she will",
    "she'll've": "she will have",
    "she's": "she is",
    "should've": "should have",
    "shouldn't": "should not",
    "shouldn't've": "should not have",
    "so've": "so have",
    "so's": "so is",
    "that'd": "that would",
    "that'd've": "that would have",
    "that's": "that is",
    "there'd": "there had",
    "there'd've": "there would have",
    "there's": "there is",
    "they'd": "they would",
    "they'd've": "they would have",
    "they'll": "they will",
    "they'll've": "they will have",
    "they're": "they are",
    "they've": "they have",
    "to've": "to have",
    "wasn't": "was not",
    "we'd": "we had",
    "we'd've": "we would have",
    "we'll": "we will",
    "we'll've": "we will have",
    "we're": "we are",
    "we've": "we have",
    "weren't": "were not",
    "what'll": "what will",
    "what'll've": "what will have",
    "what're": "what are",
    "what's": "what is",
    "what've": "what have",
    "when's": "when is",
    "when've": "when have",
    "where'd": "where did",
    "where's": "where is",
    "where've": "where have",
    "who'll": "who will",
    "who'll've": "who will have",
    "who's": "who is",
    "who've": "who have",
    "why's": "why is",
    "why've": "why have",
    "will've": "will have",
    "won't": "will not",
    "won't've": "will not have",
    "would've": "would have",
    "wouldn't": "would not",
    "wouldn't've": "would not have",
    "y'all": "you all",
    "y'alls": "you alls",
    "y'all'd": "you all would",
    "y'all'd've": "you all would have",
    "y'all're": "you all are",
    "y'all've": "you all have",
    "you'd": "you had",
    "you'd've": "you would have",
    "you'll": "you you will",
    "you'll've": "you you will have",
    "you're": "you are",
    "you've": "you have",
}



In [8]:
c_re = re.compile("(%s)" % "|".join(cList.keys()))




In [9]:
def expandContractions(text, c_re=c_re):
    def replace(match):
        return cList[match.group(0)]

    return c_re.sub(replace, text)




In [10]:
def removeHTML(x):
    html = re.compile(r"<.*?>")
    return html.sub(r"", x)




In [11]:
def dataPreprocessing(x):
    x = x.apply(lambda s: s.lower())
    x = x.apply(removeHTML)
    x = x.apply(lambda s: re.sub("@\w+", "", s))
    x = x.apply(lambda s: re.sub("'\d+", "", s))
    x = x.apply(lambda s: re.sub("\d+", "", s))
    x = x.apply(lambda s: re.sub("http\w+", "", s))
    x = x.apply(lambda s: re.sub(r"\s+", " ", s))
    x = x.apply(expandContractions)
    x = x.apply(lambda s: re.sub(r"\.+", ".", s))
    x = x.apply(lambda s: re.sub(r"\,+", ",", s))
    x = x.apply(lambda s: re.sub("\n", "", s))
    x = x.apply(lambda s: re.sub("[^\w\s]", "", s))
    x = x.apply(lambda s: s.strip())
    return x




In [12]:
x = dataPreprocessing(train_df1["full_text"])



In [13]:
x



0        dear state senator i am arguing in favor of ch...
1        in  the challenge of exploring venus the autho...
2        teachers can have a hard time telling if their...
3        using dirverless cars can be very dangrous in ...
4        in  the challenge of exploring venus the autho...
                               ...                        
15571    i think people should participate in the seago...
15572    in the article making mona lisa smile the auth...
15573    are you kidding me you actually think that fig...
15574    i feel that this could be very usefull in the ...
15575    in the article driverless cars are coming the ...
Name: full_text, Length: 15576, dtype: object

In [14]:
test_df1 = pd.read_csv(
    "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv"
)



In [15]:
x0 = dataPreprocessing(test_df1["full_text"])



In [16]:
# Bug fix: the dataset has only one target column "score".
# The original slice train_df1.iloc[:, 2:8] produced an incorrect/empty y.
y = train_df1["score"]



In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.1, random_state=123, stratify=y
)



In [18]:
y_train



13644    3
10294    3
3969     2
7603     3
8312     1
        ..
131      2
640      5
8314     2
8349     2
3371     2
Name: score, Length: 14018, dtype: int64

In [19]:
X_train



13644    hey you there would you like to be a seagoing ...
10294    the author supporting his idea by starting off...
3969     well the author is saying that venus is the mo...
7603     in the textthe challenge of exploring venus it...
8312     dear state senator i think that every person w...
                               ...                        
131      im in favor of changing to election by popular...
640      sitting in class and listening to the teacher ...
8314     the author shows many reasons that venus shoul...
8349     i would like to change the election for the pr...
3371     humans can detect what people are feeling why ...
Name: full_text, Length: 14018, dtype: object

In [20]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)



(14018,)
(14018,)
(1558,)
(1558,)


In [21]:
text_vectorizer = TfidfVectorizer(
    stop_words="english",
    sublinear_tf=False,
    strip_accents="unicode",
    binary=True,
    analyzer="word",
    token_pattern=r"\w{2,}",
    ngram_range=(1, 1),
    norm="l1",
    use_idf=False,
    smooth_idf=False,
    max_features=600000,
    min_df=30,
)



In [22]:
# Bug fix: force CSR output to avoid downstream coo_matrix incompatibilities in this environment.
X_train_features = text_vectorizer.fit_transform(X_train).tocsr()



In [23]:
X_train_features



<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1294023 stored elements and shape (14018, 4313)>

In [24]:
# Bug fix: remove MaxAbsScaler which triggers .A access on coo_matrix under this stack.
# Core logic preserved: still TF-IDF features + RBF SVC classification.
from sklearn.svm import SVC

test_features = text_vectorizer.transform(X_test).tocsr()

clf = SVC(
    C=1.75,
    kernel="rbf",
    gamma="scale",
    decision_function_shape="ovr",
    random_state=123,
    tol=1e-5,
    shrinking=True,
    verbose=True,
    break_ties=True,
)
clf.fit(X_train_features, y_train.values.ravel())
y_pred = clf.predict(test_features)



[LibSVM]........*.*
optimization finished, #iter = 9588
obj = -1715.592908, rho = 0.039014
nSV = 3699, nBSV = 599
......*.*
optimization finished, #iter = 7675
obj = -1219.963128, rho = -1.136563
nSV = 2827, nBSV = 331
...*.*
optimization finished, #iter = 4289
obj = -576.747264, rho = -2.159723
nSV = 1502, nBSV = 142
.*
optimization finished, #iter = 1754
obj = -209.304447, rho = -2.685868
nSV = 624, nBSV = 49
*
optimization finished, #iter = 627
obj = -83.224458, rho = -2.822940
nSV = 235, nBSV = 11
................*....*
optimization finished, #iter = 20011
obj = -4472.976463, rho = -2.088902
nSV = 7207, nBSV = 1693
.......*.*
optimization finished, #iter = 8707
obj = -1656.881272, rho = -3.709334
nSV = 3042, nBSV = 626
..*
optimization finished, #iter = 2706
obj = -396.186487, rho = -4.176831
nSV = 935, nBSV = 109
*
optimization finished, #iter = 765
obj = -116.387008, rho = -3.532855
nSV = 293, nBSV = 30
.............*...*
optimization finished, #iter = 16101
obj = -4720.563087, r

In [25]:
print(confusion_matrix(y_test.values.ravel(), y_pred.ravel()))



[[ 39  55  15   2   1   0]
 [  8 267 134  15   1   0]
 [  1 104 359  98   1   0]
 [  0   3 119 220  14   0]
 [  0   0   4  46  38   0]
 [  0   0   0   5   9   0]]


In [26]:
print(classification_report(y_test.values.ravel(), y_pred.ravel()))



              precision    recall  f1-score   support

           1       0.81      0.35      0.49       112
           2       0.62      0.63      0.63       425
           3       0.57      0.64      0.60       563
           4       0.57      0.62      0.59       356
           5       0.59      0.43      0.50        88
           6       0.00      0.00      0.00        14

    accuracy                           0.59      1558
   macro avg       0.53      0.44      0.47      1558
weighted avg       0.60      0.59      0.59      1558



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [27]:
kappa = cohen_kappa_score(y_test.values.ravel(), y_pred.ravel(), weights="quadratic")
print("Cohen's kappa score: ", kappa)



Cohen's kappa score:  0.7285227245703809


In [28]:
st_features = text_vectorizer.transform(x0).tocsr()
test_predictions = clf.predict(st_features)



In [29]:
# Bug fix: ensure submission aligns to test essay_id and output is a valid .csv.
submission = pd.read_csv(
    "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/sample_submission.csv"
)
submission["score"] = test_predictions.astype(int)
submission.to_csv("submission.csv", index=False)
print(submission.head())
print("Wrote submission.csv with shape:", submission.shape)

  essay_id  score
0  d550b2d      2
1  0c10954      2
2  ef04816      3
3  88ab8d9      2
4  cee8c0f      2
Wrote submission.csv with shape: (1731, 2)
